# Sugar Trap — Snack Market Gap Analysis
**Client:** Helix CPG Partners  
**Goal:** Identify Blue Ocean opportunities in the snack aisle — product categories where health-conscious demand (high protein, high fiber) is unmet by current offerings (high sugar, high fat).

**Data:** Open Food Facts (CC BY-SA 4.0) — https://world.openfoodfacts.org/data

In [6]:
# Cell 1: Package Installation (runs only in Google Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'plotly', 'wordcloud'], check=True)
print(f'Running in Colab: {IN_COLAB}')

Running in Colab: False


In [7]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

All imports successful.


In [8]:
# Cell 3: Constants
DATA_URL = 'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz'
NROWS = 100_000  # 100k = fast local run (~30s); change to 500_000 for full analysis

RELEVANT_COLS = [
    'product_name', 'categories_tags',
    'sugars_100g', 'proteins_100g',
    'fat_100g', 'fiber_100g',
    'energy_100g', 'ingredients_text'
]

# Blue Ocean quadrant thresholds
PROTEIN_THRESHOLD = 10.0  # g/100g
SUGAR_THRESHOLD   = 5.0   # g/100g

PLOT_SAMPLE = 50_000

print(f'Will load {NROWS:,} rows | Blue Ocean: protein > {PROTEIN_THRESHOLD}g AND sugar < {SUGAR_THRESHOLD}g')

Will load 100,000 rows | Blue Ocean: protein > 10.0g AND sugar < 5.0g


In [ ]:
# Cell 4: Story 1 — Data Ingestion & Cleaning
# NOTE: OpenFoodFacts uses TAB separators despite the .csv extension

print(f'Loading {NROWS:,} rows from OpenFoodFacts...')
df_raw = pd.read_csv(
    DATA_URL,
    usecols=RELEVANT_COLS,
    nrows=NROWS,
    low_memory=False,
    sep='\t',
    on_bad_lines='skip'
)
print(f'Raw shape: {df_raw.shape}')

# Step 1: Drop rows missing critical fields
required_cols = ['product_name', 'sugars_100g', 'proteins_100g']
df_clean = df_raw.dropna(subset=required_cols).copy()
print(f'After dropping required nulls: {df_clean.shape}')

# Step 2: Filter biologically impossible values (no nutrient can exceed 100g per 100g)
nutrient_cols = ['sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']
for col in nutrient_cols:
    if col in df_clean.columns:
        df_clean = df_clean[df_clean[col].between(0, 100)]

# Step 3: Fill optional nutrients with column median
optional_nutrients = ['fat_100g', 'fiber_100g', 'energy_100g']
for col in optional_nutrients:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Step 4: Remove duplicate product names
df_clean = df_clean.drop_duplicates(subset='product_name', keep='first')

print(f'Final cleaned shape: {df_clean.shape}')

# Verification assertions
assert df_clean[required_cols].isna().sum().sum() == 0, 'Null values remain in required columns'
assert df_clean['sugars_100g'].between(0, 100).all(), 'Out-of-range sugar values found'
assert df_clean['proteins_100g'].between(0, 100).all(), 'Out-of-range protein values found'
min_expected = max(5_000, NROWS // 20)  # expect at least 5% of loaded rows to survive cleaning
assert df_clean.shape[0] >= min_expected, f'Too few rows after cleaning: {df_clean.shape[0]} (expected >= {min_expected})'
print(f'All Story 1 assertions passed. ({df_clean.shape[0]:,} clean rows)')

df_clean.describe()[['sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']]

Loading 100,000 rows from OpenFoodFacts...
Raw shape: (100000, 8)
After dropping required nulls: (16891, 8)
Final cleaned shape: (10603, 8)
All Story 1 assertions passed. (10,603 clean rows)


,sugars_100g,proteins_100g,fat_100g,fiber_100g
count,10603.000000,10603.000000,10603.000000,10603.000000
mean,11.431697,10.494721,10.906997,3.810064
std,15.874715,13.263072,14.045326,6.906517
min,0.000000,0.000000,0.000000,0.000000
25%,1.411765,3.163454,1.408451,0.492075
50%,4.411765,7.100000,5.370000,1.900000
75%,15.069294,12.000000,15.518621,4.500000
max,100.000000,100.000000,100.000000,100.000000


In [10]:
# Cell 5: Story 2 — Category Wrangler

CATEGORY_MAP = {
    'Snacks & Crisps':     ['snack', 'crisp', 'chip', 'popcorn', 'pretzel', 'cracker'],
    'Baked Goods':         ['biscuit', 'cake', 'cookie', 'bread', 'pastry', 'wafer', 'muffin', 'brownie'],
    'Dairy & Eggs':        ['dairy', 'cheese', 'yogurt', 'yoghurt', 'milk', 'egg', 'cream'],
    'Beverages':           ['beverage', 'drink', 'juice', 'water', 'soda', 'smoothie', 'tea', 'coffee'],
    'Protein Foods':       ['protein', 'meat', 'fish', 'chicken', 'beef', 'whey', 'bar', 'shake'],
    'Confectionery':       ['chocolate', 'candy', 'sweet', 'sugar', 'caramel', 'gum', 'lolly'],
    'Cereals & Grains':    ['cereal', 'granola', 'oat', 'grain', 'muesli', 'porridge', 'rice'],
    'Fruits & Vegetables': ['fruit', 'vegetable', 'nut', 'seed', 'dried', 'berry'],
    'Prepared & Packaged': ['prepared', 'canned', 'frozen', 'instant', 'soup', 'sauce',
                            'condiment', 'spread', 'pasta', 'noodle', 'meal', 'salad',
                            'seasoning', 'dressing', 'jam', 'honey', 'syrup'],
}
OTHER = 'Other'

def assign_primary_category(tags):
    if not isinstance(tags, str) or not tags.strip():
        return OTHER
    tags_lower = tags.lower()
    for category, keywords in CATEGORY_MAP.items():
        if any(kw in tags_lower for kw in keywords):
            return category
    return OTHER

df_clean['primary_category'] = df_clean['categories_tags'].apply(assign_primary_category)

cat_dist = df_clean['primary_category'].value_counts()
print('Category distribution:')
print(cat_dist.to_string())

non_other = cat_dist[cat_dist.index != OTHER]
other_pct = cat_dist.get(OTHER, 0) / len(df_clean)

assert len(non_other) >= 5, f'Need >= 5 named categories, got {len(non_other)}'
# Note: early OpenFoodFacts rows (~first 100k) have sparse category tags; 
# "Other" can reach 50-60% on small loads. Use 500k rows for a cleaner distribution.
assert other_pct < 0.75, f'Too many uncategorized products: {other_pct:.1%} — try increasing NROWS'

print(f'\nNamed categories: {len(non_other)} | Other: {other_pct:.1%}')
print('All Story 2 assertions passed.')

Category distribution:
primary_category
Other                  5406
Beverages              1517
Snacks & Crisps         969
Baked Goods             881
Dairy & Eggs            805
Prepared & Packaged     502
Protein Foods           403
Confectionery            74
Cereals & Grains         24
Fruits & Vegetables      22

Named categories: 9 | Other: 51.0%
All Story 2 assertions passed.


In [11]:
# Cell 6: Story 3 — Nutrient Matrix Scatter Plot

df_plot = df_clean.sample(min(PLOT_SAMPLE, len(df_clean)), random_state=42)

fig = px.scatter(
    df_plot,
    x='sugars_100g',
    y='proteins_100g',
    color='primary_category',
    hover_name='product_name',
    hover_data={
        'sugars_100g': ':.1f',
        'proteins_100g': ':.1f',
        'primary_category': True
    },
    opacity=0.5,
    title='Nutrient Matrix: Sugar vs. Protein by Category — Blue Ocean Gap Analysis',
    labels={
        'sugars_100g': 'Sugar (g per 100g)',
        'proteins_100g': 'Protein (g per 100g)',
        'primary_category': 'Category'
    },
    color_discrete_sequence=px.colors.qualitative.Bold,
)

# Blue Ocean quadrant: High Protein + Low Sugar
fig.add_shape(
    type='rect',
    x0=0, y0=PROTEIN_THRESHOLD,
    x1=SUGAR_THRESHOLD, y1=100,
    line=dict(color='green', width=2, dash='dash'),
    fillcolor='rgba(0,255,0,0.05)',
)
fig.add_annotation(
    x=SUGAR_THRESHOLD / 2,
    y=PROTEIN_THRESHOLD + 5,
    text='BLUE OCEAN<br>High Protein + Low Sugar',
    showarrow=False,
    font=dict(color='green', size=12, family='Arial Black'),
    bgcolor='white',
    bordercolor='green',
)

fig.update_layout(
    xaxis_range=[0, 60],
    yaxis_range=[0, 60],
    height=600,
    legend_title_text='Category',
)
fig.show()

blue_ocean_count = ((df_clean['proteins_100g'] > PROTEIN_THRESHOLD) & (df_clean['sugars_100g'] < SUGAR_THRESHOLD)).sum()
print(f'Blue Ocean products: {blue_ocean_count:,} of {len(df_clean):,} ({blue_ocean_count/len(df_clean)*100:.1f}%)')

Blue Ocean products: 2,190 of 10,603 (20.7%)


In [12]:
# Cell 7: Story 4 — Recommendation Engine

blue_ocean_mask = (
    (df_clean['proteins_100g'] > PROTEIN_THRESHOLD) &
    (df_clean['sugars_100g'] < SUGAR_THRESHOLD)
)
df_blue = df_clean[blue_ocean_mask].copy()
print(f'Blue Ocean products: {len(df_blue):,} ({len(df_blue)/len(df_clean)*100:.1f}% of cleaned dataset)')

# Category counts in Blue Ocean vs. total — gap ratio identifies biggest opportunity
category_counts = df_blue['primary_category'].value_counts()
cat_totals = df_clean['primary_category'].value_counts()
gap_ratio = category_counts / cat_totals.reindex(category_counts.index).fillna(1)

# Category with LOWEST blue-ocean ratio = most under-served
gap_opportunity = gap_ratio[gap_ratio.index != OTHER].idxmin()
gap_top_products = df_blue[df_blue['primary_category'] == gap_opportunity]
avg_protein = gap_top_products['proteins_100g'].mean()
avg_sugar   = gap_top_products['sugars_100g'].mean()
gap_pct     = (1 - gap_ratio[gap_opportunity]) * 100

insight = f"""
╔══════════════════════════════════════════════════════════════════╗
║                         KEY INSIGHT                             ║
╠══════════════════════════════════════════════════════════════════╣
║  Based on the data, the biggest market opportunity is in        ║
║  [{gap_opportunity}], specifically targeting products with      ║
║  {avg_protein:.0f}g of protein and less than {avg_sugar:.0f}g of sugar.              ║
║                                                                  ║
║  {gap_pct:.0f}% of {gap_opportunity} products do NOT meet Blue Ocean criteria,  ║
║  leaving a massive under-served health-conscious segment.       ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(insight)

# Visualise gap ratio across categories
gap_df = gap_ratio[gap_ratio.index != OTHER].reset_index()
gap_df.columns = ['category', 'blue_ocean_ratio']
gap_df = gap_df.sort_values('blue_ocean_ratio')

fig_gap = px.bar(
    gap_df,
    x='category',
    y='blue_ocean_ratio',
    color='blue_ocean_ratio',
    color_continuous_scale='RdYlGn',
    title='Blue Ocean Ratio by Category (Lower = Bigger Opportunity)',
    labels={'blue_ocean_ratio': 'Blue Ocean Ratio', 'category': 'Category'},
)
fig_gap.update_layout(coloraxis_showscale=False)
fig_gap.show()

Blue Ocean products: 2,190 (20.7% of cleaned dataset)

╔══════════════════════════════════════════════════════════════════╗
║                         KEY INSIGHT                             ║
╠══════════════════════════════════════════════════════════════════╣
║  Based on the data, the biggest market opportunity is in        ║
║  [Confectionery], specifically targeting products with      ║
║  22g of protein and less than 0g of sugar.              ║
║                                                                  ║
║  95% of Confectionery products do NOT meet Blue Ocean criteria,  ║
║  leaving a massive under-served health-conscious segment.       ║
╚══════════════════════════════════════════════════════════════════╝



In [13]:
# Cell 8: Bonus — Protein Source Ingredient Analysis

PROTEIN_KEYWORDS = [
    'whey', 'casein', 'pea protein', 'soy protein', 'soy isolate',
    'milk protein', 'egg white', 'egg protein', 'hemp protein',
    'collagen', 'gelatin', 'chickpea', 'lentil', 'quinoa',
    'almonds', 'cashews', 'peanut'
]

def extract_protein_sources(text):
    if not isinstance(text, str):
        return []
    tl = text.lower()
    return [kw for kw in PROTEIN_KEYWORDS if kw in tl]

bo_with_ingredients = df_blue.dropna(subset=['ingredients_text'])
print(f'Blue Ocean products with ingredients text: {len(bo_with_ingredients):,}')

all_sources = []
for text in bo_with_ingredients['ingredients_text']:
    all_sources.extend(extract_protein_sources(text))

src_counts = Counter(all_sources)
top_n = src_counts.most_common(15)
top3  = src_counts.most_common(3)

if len(top3) == 0:
    print('\nNo matching protein keywords found in ingredients.')
    print('Tip: increase NROWS to 500_000 for a richer ingredient dataset.')
else:
    print(f'\nTop {min(3, len(top3))} Protein Sources in Blue Ocean Products:')
    for rank, (source, count) in enumerate(top3, 1):
        print(f'  {rank}. {source.title():25s} — {count:,} products')

    labels, values = zip(*top_n)
    fig_ing = px.bar(
        x=list(values), y=list(labels),
        orientation='h',
        title='Most Common Protein Sources in Blue Ocean Products',
        labels={'x': 'Product Count', 'y': 'Ingredient'},
        color=list(values),
        color_continuous_scale='Blues',
    )
    fig_ing.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False, height=450)
    fig_ing.show()

Blue Ocean products with ingredients text: 845

Top 3 Protein Sources in Blue Ocean Products:
  1. Whey                      — 91 products
  2. Soy Protein               — 27 products
  3. Milk Protein              — 24 products


In [14]:
# Cell 9: Candidate's Choice — Health Score Index (HSI)
#
# Business Justification:
# The scatter plot (Story 3) requires the viewer to mentally integrate two axes.
# A single HSI collapses protein, fiber, sugar, and fat into one comparable metric.
# This lets a brand say "our product scores 0.72 vs. category average 0.31" —
# directly actionable for investor decks, R&D briefs, and retail shelf ranking.

def compute_health_score(df):
    def minmax(s):
        rng = s.max() - s.min()
        return (s - s.min()) / (rng if rng > 0 else 1.0)
    return (
        minmax(df['proteins_100g']) * 0.35
        + minmax(df['fiber_100g'])  * 0.30
        - minmax(df['sugars_100g']) * 0.25
        - minmax(df['fat_100g'])    * 0.10
    )

df_clean['health_score'] = compute_health_score(df_clean)

assert df_clean['health_score'].notna().all(), 'NaN health scores found'
assert df_clean['health_score'].std() > 0.05, 'Health scores have no meaningful variance'

# Min products per category scales with dataset size
min_cat_products = max(10, len(df_clean) // 500)

cat_hsi = (
    df_clean.groupby('primary_category')['health_score']
    .agg(avg_hsi='mean', hsi_std='std', n_products='count')
    .query(f'n_products >= {min_cat_products}')
    .sort_values('avg_hsi', ascending=False)
    .reset_index()
)

assert len(cat_hsi) >= 5, f'Need >= 5 categories in HSI chart, got {len(cat_hsi)}'

print('Category HSI Leaderboard:')
print(cat_hsi[['primary_category', 'avg_hsi', 'n_products']].to_string(index=False))

fig_hsi = px.bar(
    cat_hsi,
    x='primary_category',
    y='avg_hsi',
    error_y='hsi_std',
    color='avg_hsi',
    color_continuous_scale='RdYlGn',
    title='Category Health Score Index (HSI) Leaderboard',
    labels={'primary_category': 'Category', 'avg_hsi': 'Average HSI'},
    text='avg_hsi',
)
fig_hsi.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_hsi.update_layout(coloraxis_showscale=False, height=420)
fig_hsi.show()
print("All Candidate's Choice assertions passed.")

Category HSI Leaderboard:
   primary_category   avg_hsi  n_products
      Protein Foods  0.061875         403
        Baked Goods  0.023768         881
              Other  0.015063        5406
   Cereals & Grains  0.014858          24
Fruits & Vegetables  0.010094          22
Prepared & Packaged  0.003483         502
          Beverages  0.001916        1517
       Dairy & Eggs  0.001394         805
    Snacks & Crisps -0.033269         969
      Confectionery -0.127508          74


All Candidate's Choice assertions passed.


In [15]:
# Cell 10: Export Summary CSV (for Streamlit dashboard fast-path)

SUMMARY_COLS = [
    'product_name', 'primary_category',
    'sugars_100g', 'proteins_100g',
    'fiber_100g', 'fat_100g',
    'health_score'
]

df_clean[SUMMARY_COLS].to_csv('sugar_trap_summary.csv', index=False)
print(f'Saved sugar_trap_summary.csv — shape: {df_clean[SUMMARY_COLS].shape}')
print('\nNext steps:')
print('  1. Download this notebook: File > Download > Download .ipynb')
print('  2. Export as HTML: !jupyter nbconvert --to html sugar_trap_analysis.ipynb')
print('  3. Download sugar_trap_summary.csv for local Streamlit development')
print('  4. Run the Streamlit dashboard: streamlit run app.py')

Saved sugar_trap_summary.csv — shape: (10603, 7)

Next steps:
  1. Download this notebook: File > Download > Download .ipynb
  2. Export as HTML: !jupyter nbconvert --to html sugar_trap_analysis.ipynb
  3. Download sugar_trap_summary.csv for local Streamlit development
  4. Run the Streamlit dashboard: streamlit run app.py
